# NLP From Scratch: Classifying Names with a Character-Level RNN

In [1]:
# https://docs.pytorch.org/tutorials/intermediate/char_rnn_classification_tutorial.html

In [2]:
# Mount the drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
import string
import unicodedata

# We can use "_" to represent an out-of-vocabulary character, that is, any character we are not handling in our model
allowed_characters = string.ascii_letters + " .,;'" + "_"
n_letters = len(allowed_characters)

# Turn a Unicode string to plain ASCII, thanks to https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in allowed_characters
    )

In [5]:
print (f"converting 'Ślusàrski' to {unicodeToAscii('Ślusàrski')}")

converting 'Ślusàrski' to Slusarski


In [6]:
allowed_characters

"abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ .,;'_"

In [7]:
def letterToIndex(letter):
  if letter not in allowed_characters:
    return allowed_characters.find("_")
  else:
    return allowed_characters.find(letter)

In [12]:
def lineToTensor(line):
  tensor = torch.zeros(len(line), len(allowed_characters))
  for i, letter in enumerate(line):
    tensor[i][letterToIndex(letter)] = 1
  tensor.unsqueeze_(1)
  return tensor

In [13]:
tensor = lineToTensor("abc_")

In [14]:
tensor.shape

torch.Size([4, 1, 58])

In [15]:
path = "/content/drive/MyDrive/Colab Notebooks/PyTorch Learning/nlp_1_data/names/"

In [17]:
## Create the Dataset

In [24]:
import torch
from torch.utils.data import Dataset
import time
import os
import glob

class NamesDataset(Dataset):
  def __init__(self, data_dir):
    self.data_dir = data_dir #for provenance of the dataset
    self.load_time = time.localtime #for provenance of the dataset
    labels_set = set() #set of all classes

    self.data = []
    self.data_tensors = []
    self.labels = []
    self.labels_tensors = []

    text_files = glob.glob(os.path.join(data_dir, '*.txt'))
    for filename in text_files:
      label = os.path.splitext(os.path.basename(filename))[0]
      labels_set.add(label)
      lines = open(filename, encoding='utf-8').read().strip().split('\n')
      for name in lines:
        self.data.append(name)
        self.data_tensors.append(lineToTensor(name))
        self.labels.append(label)

    # Cache the tensor representation of the labels
    self.labels_uniq = list(labels_set)
    for idx in range(len(self.labels)):
        temp_tensor = torch.tensor([self.labels_uniq.index(self.labels[idx])], dtype=torch.long)
        self.labels_tensors.append(temp_tensor)

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    data_item = self.data[idx]
    data_label = self.labels[idx]
    data_tensor = self.data_tensors[idx]
    label_tensor = self.labels_tensors[idx]

    return label_tensor, data_tensor, data_label, data_item


In [28]:
alldata = NamesDataset("/content/drive/MyDrive/Colab Notebooks/PyTorch Learning/nlp_1_data/names")

In [29]:
len(alldata)

20074

In [37]:
label_tensor, data_tensor, data_label, data_item = alldata[1]

In [42]:
label_tensor

tensor([0])

In [39]:
data_tensor.shape

torch.Size([6, 1, 58])

In [40]:
data_label

'Irish'

In [41]:
data_item

'Ahearn'

In [43]:
train_set, test_set = torch.utils.data.random_split(alldata, [.85, .15], generator=torch.Generator(device=device).manual_seed(2024))

In [44]:
print(f"train examples = {len(train_set)}, validation examples = {len(test_set)}")

train examples = 17063, validation examples = 3011


In [45]:
## Create the neural network

In [46]:
import torch.nn.functional as F

In [104]:
class CharRNN(torch.nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()

    self.rnn = torch.nn.RNN(input_size, hidden_size)
    self.linear_layer = torch.nn.Linear(hidden_size, output_size)
    self.softmax = torch.nn.LogSoftmax(dim=1)

  def forward(self, x):
    rnn_out, hidden = self.rnn(x)
    output = self.linear_layer(hidden).squeeze(1)
    output = self.softmax(output)

    return output

In [105]:
rnn = CharRNN(n_letters, 128, len(alldata.labels_uniq))

In [106]:
output = rnn(data_tensor)

In [107]:
data_tensor.shape

torch.Size([6, 1, 58])

In [108]:
output.shape

torch.Size([1, 18])

In [109]:
output

tensor([[-2.9320, -2.8394, -2.7014, -2.9629, -2.8300, -3.0195, -2.9922, -2.8767,
         -2.9364, -2.9392, -2.8729, -2.9338, -2.9657, -2.9024, -2.9055, -2.9486,
         -2.7069, -2.8278]], grad_fn=<LogSoftmaxBackward0>)

In [114]:
def label_from_output(output, output_labels):
  pred_idx = torch.argmax(output)
  return output_labels[pred_idx], pred_idx.item()

In [115]:
input = lineToTensor('LOL')
output = rnn(input)
print(output)
print(label_from_output(output, alldata.labels_uniq))

tensor([[-2.9361, -2.8509, -2.7611, -2.9079, -2.8547, -3.0153, -2.9500, -2.9070,
         -2.8547, -2.8835, -2.9216, -2.8667, -2.8992, -2.9228, -2.9489, -2.9650,
         -2.7964, -2.8190]], grad_fn=<LogSoftmaxBackward0>)
('Dutch', 2)


In [116]:
# Forward pass is ...